- **Fase 2 — Tre baseline** di anomaly detection:
  - statistica classica (STL + soglia su residui via MAD)
  - density-based (IsolationForest / LOF con feature engineered)
  - forecasting-based (gradient boosting con lag + meteo come feature)

Tutto il dettaglio è nel documento di progetto.

### Choice of the coils 

Per scegliere le spire, dvido la città in rettangoli e scelgo per ciascuna zona una spira più vicina al centro del rettangolo. In totale ci sono 100 aree della città e quindi 100 spire.
 Parameters
    ----------
    df : DataFrame
        Deve contenere latitudine e longitudine
    n_lat : int
        numero di celle in latitudine
    n_lon : int
        numero di celle in longitudine
    mode : str
        - "centroid": prende il sensore più vicino al centro cella
        - "random": prende un sensore casuale per cella
    """

In [2]:
import pandas as pd

df = pd.read_parquet('../data/processed/dataset_finale_2024-01-01_2025-09-30.parquet')
df.head()

,data,id_uni,chiave,nome_via,direzione,longitudine,latitudine,fascia_oraria,conteggio_veicoli,ora,...,giorno_settimana,ora_del_giorno,weekend,pioggia,accuratezza,data_giorno,festivo_nazionale,nome_festivita,festa_locale,tipo_giorno
0,2024-01-01,63,9,VIA GIUSEPPE PETRONI,S,11.352084,44.494329,00_00_01_00,9,0,...,Monday,0,False,False,100.0,2024-01-01,True,Capodanno,False,festivo
1,2024-01-01,63,9,VIA GIUSEPPE PETRONI,S,11.352084,44.494329,01_00_02_00,12,1,...,Monday,1,False,False,100.0,2024-01-01,True,Capodanno,False,festivo
2,2024-01-01,63,9,VIA GIUSEPPE PETRONI,S,11.352084,44.494329,02_00_03_00,12,2,...,Monday,2,False,False,100.0,2024-01-01,True,Capodanno,False,festivo
3,2024-01-01,63,9,VIA GIUSEPPE PETRONI,S,11.352084,44.494329,03_00_04_00,10,3,...,Monday,3,False,True,100.0,2024-01-01,True,Capodanno,False,festivo
4,2024-01-01,63,9,VIA GIUSEPPE PETRONI,S,11.352084,44.494329,04_00_05_00,10,4,...,Monday,4,False,True,100.0,2024-01-01,True,Capodanno,False,festivo


In [4]:
df_filt = df[df['chiave'].isin([9,  10,  13,  32,  34,  36,  37,  39,  46,  47,  58,  61,  64,])]

In [5]:
df.columns

Index(['data', 'id_uni', 'chiave', 'nome_via', 'direzione', 'longitudine',
       'latitudine', 'fascia_oraria', 'conteggio_veicoli', 'ora', 'timestamp',
       'temperature_2m', 'precipitation', 'rain', 'wind_speed_10m',
       'weather_code', 'tempo', 'giorno_settimana', 'ora_del_giorno',
       'weekend', 'pioggia', 'accuratezza', 'data_giorno', 'festivo_nazionale',
       'nome_festivita', 'festa_locale', 'tipo_giorno'],
      dtype='str')

In [6]:
veicoli_medi_per_spira = (
    df.groupby(['chiave', 'data'])['conteggio_veicoli']
    .sum()
    .groupby('chiave')
    .mean()
    .rename("veicoli_medi")
)
# merge con il dataframe delle coordinate
df_mappa = df.merge(veicoli_medi_per_spira, on='chiave')

In [7]:
import plotly.express as px
import plotly.io as pio
import os
os.environ["BROWSER"] = "chrome" 
pio.renderers.default = "browser"

def map_coils(df):
    fig = px.scatter_map(
        df[['chiave', 'latitudine', 'longitudine','veicoli_medi']].drop_duplicates(subset='chiave'),
        lat="latitudine",
        lon="longitudine",
        hover_name="chiave",
        # size='veicoli_medi',
        zoom=11,
        height=600
    )

    fig.update_layout(
        mapbox_style="open-street-map",  # non richiede token
        margin={"r":0, "t":0, "l":0, "b":0}
    )

    fig.write_html("mappa.html")
map_coils(df_mappa)

Let's undersample the coils

In [8]:
import numpy as np
import pandas as pd

def sottocampiona_griglia(
    df: pd.DataFrame,
    lat_col: str = "latitudine",
    lon_col: str = "longitudine",
    id_col: str = "chiave",
    n_lat: int = 10,
    n_lon: int = 10,
    mode: str = "centroid"  # "centroid" o "random"
) -> pd.DataFrame:
    """
    Sottocampiona sensori usando una griglia geografica.

    Parameters
    ----------
    df : DataFrame
        Deve contenere latitudine e longitudine
    n_lat : int
        numero di celle in latitudine
    n_lon : int
        numero di celle in longitudine
    mode : str
        - "centroid": prende il sensore più vicino al centro cella
        - "random": prende un sensore casuale per cella
    """

    df = df.copy()

    lat_min, lat_max = df[lat_col].min(), df[lat_col].max()
    lon_min, lon_max = df[lon_col].min(), df[lon_col].max()

    # dimensione celle
    lat_bins = np.linspace(lat_min, lat_max, n_lat + 1)
    lon_bins = np.linspace(lon_min, lon_max, n_lon + 1)

    # assegna cella
    df["lat_bin"] = np.digitize(df[lat_col], lat_bins) - 1
    df["lon_bin"] = np.digitize(df[lon_col], lon_bins) - 1

    selected = []

    for (_, group) in df.groupby(["lat_bin", "lon_bin"]):

        if len(group) == 0:
            continue

        if mode == "random":
            selected.append(group.sample(1))

        elif mode == "centroid":
            # centro cella
            lat_center = group[lat_col].mean()
            lon_center = group[lon_col].mean()

            # distanza dal centro
            dist = (group[lat_col] - lat_center)**2 + (group[lon_col] - lon_center)**2

            selected.append(group.loc[[dist.idxmin()]])

        else:
            raise ValueError("mode deve essere 'centroid' o 'random'")

    return pd.concat(selected).reset_index(drop=True)

df_usamp = sottocampiona_griglia(df_mappa, n_lat=10, n_lon=10,)
map_coils(df_usamp)

### Coils accuracy management

Percentage of time in which accuracy is lower than 75% for each coil. 

In [18]:
df.groupby('id_uni')['accuratezza'].apply(lambda x: round((x < 75).sum()/len(x)*100, 2))

id_uni
20      6.10
21     35.80
22      2.98
23     20.69
24      5.37
       ...  
945     0.35
946     7.59
947     1.54
948     0.31
949     1.30
Name: accuratezza, Length: 100, dtype: float64